# Graph Convolutional Neural Networks

## Introduction
* A convolutional layer condenses the information all around a given pixel using a kernl (downscaling)

![CNN Diagram](https://miro.medium.com/v2/resize:fit:1066/format:webp/0*8ThP2FaZhZy20RnB.png)
[Image Source](https://medium.com/@thepyprogrammer/2d-image-convolution-with-numpy-with-a-handmade-sliding-window-view-946c4acb98b4)

* A GCN layer also condenses information from its neighbors, and also from itself

![GCN](https://miro.medium.com/v2/resize:fit:1400/format:webp/0*nf7LRRYTlG1EI0i-)
[Image Source](https://a-j.gitbook.io/geometric-deep-learning/graphs-ii)

## Inputs to GCN
* Feature Matrix, $X$
* adjacency Matrix, $A$, a matrix of ones and zeros
    * A 1 in location $a_{ij}$ represents a connection between nodes i and j

## Matematical Overview
* $AX$ is the sum of all neighbors of each node, however, to make sure the node also learns abou itself we need to make all diagonal elements of the adjacency matrix 1 as well:
    * $\hat{A} = A + \textbf{I}$
* Then: $(\hat{A}X)_i = \sum^N_i\hat{a}_{ij}\textbf{x}_j = \sum_{j\in N_i}x_j+x_i$
* *BUT* we now need to normalize the summations to account for some nodes having many neighbors and other having few.
* Create a degree matrix, $D$, which is diagonal and the diagonal elements represent the node degree (number of connections)
* We want to normalize the rows and columns of $\hat{A}X$, so we need to perform the following operation:
$$D^{-\frac{1}{2}}\hat{X}D^{-\frac{1}{2}}X$$
* One final step to get from the above aggregation of node features to an output, need to add a set of weights, $W$, so the network can learn and need to add an activation function, $f$, which is usually ReLU:
$$H = f(D^{-\frac{1}{2}}\hat{X}D^{-\frac{1}{2}}W)$$

## Simple PyTorch Implementation From Scratch

The below code is modified from [this code](https://medium.com/@jrosseruk/demystifying-gcns-a-step-by-step-guide-to-building-a-graph-convolutional-network-layer-in-pytorch-09bf2e788a51) posted on Medium by J. Rosser.

In [7]:
#############
## IMPORTS ##
#############

import torch
import torch.nn as nn
import torch.nn.functional as F

In [8]:
############################
## GCN LAYER FROM SCRATCH ##
############################
class GCNLayer(nn.Module):
    """
        GCN layer

        Args:
            input_dim (int): Dimension of the input
            output_dim (int): Dimension of the output (a softmax distribution)
            A (torch.Tensor): 2D adjacency matrix
    """

    def __init__(self, input_dim: int, output_dim: int, A: torch.Tensor):
        super(GCNLayer, self).__init__()
        """
        Inputs:
            input_dim: The number of features in the input node feature matrix X. This is the dimensionality of the input node features.
            output_dim: The number of features in the output node feature matrix H. This is the dimensionality of the output node features after applying the GCN layer.
            A: The adjacency matrix of the graph. This is a square matrix where the entry A[i][j] is 1 if there is an edge from node i to node j, and 0 otherwise. The adjacency matrix is used to determine how information is propagated between nodes in the graph. 
        Outputs:
            None 
        Initializes the GCN layer by setting up the necessary matrices and parameters for the forward pass. This includes creating the normalized adjacency matrix and initializing the weight matrix W that will be learned during training.
        """
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.A = A

        # A_hat = A + I
        self.A_hat = self.A + torch.eye(self.A.size(0))

        # Create diagonal degree matrix D
        # Make a matrix of ones with the same shape as A
        self.ones = torch.ones(input_dim, input_dim)
        # Multiply A by the matrix of ones to get the degree of each node (number of neighbors)
        self.D = torch.matmul(self.A.float(), self.ones.float())
        # Extract the diagonal elements
        self.D = torch.diag(self.D)
        # Create a new tensor with the diagonal elements and zeros elsewhere
        self.D = torch.diag_embed(self.D)
        
        # Create D^{-1/2} as this is what we use in the GCN layer
        # equations
        self.D_neg_sqrt = torch.diag_embed(torch.diag(torch.pow(self.D, -0.5)))
        
        # Initialise the weight matrix as a parameter. This is what we will learn during training. Here we are using a simple random initialization, but in practice you might want to use a more sophisticated initialization method (e.g., Xavier or Kaiming initialization) for better convergence.
        self.W = nn.Parameter(torch.rand(input_dim, output_dim))

    def forward(self, X: torch.Tensor):
        """
        Inputs:
            X: The input node feature matrix. This is a matrix where each row corresponds to a node in the graph, and each column corresponds to a feature of that node. The shape of X is (num_nodes, input_dim).
        Outputs:
            H: The output node feature matrix after applying the GCN transformation. This is a matrix where each row corresponds to a node in the graph, and each column corresponds to a feature of that node after the GCN layer. The shape of H is (num_nodes, output_dim).
        Forward pass of the GCN layer. This is where the actual computation happens. The input X is the node feature matrix, and the output H is the new node feature matrix after applying the GCN transformation.
        """

        # D^-1/2 * (A_hat * D^-1/2)
        support_1 = torch.matmul(self.D_neg_sqrt, torch.matmul(self.A_hat, self.D_neg_sqrt))
        
        # (D^-1/2 * A_hat * D^-1/2) * (X * W)
        support_2 = torch.matmul(support_1, torch.matmul(X, self.W))
        
        # ReLU(D^-1/2 * A_hat * D^-1/2 * X * W)
        H = F.relu(support_2)

        return H

In [9]:

###################################
## EXAMPLE WITH THREE NODE GRAPH ##
####################################

# Test the forward pass of the GCN layer with a simple example graph. This is a small graph with 3 nodes and an adjacency matrix that represents the connections between the nodes. The input feature matrix X has 3 features for each node, and we want to transform it to an output feature matrix with 2 features for each node using the GCN layer. There is no training here, we are just testing the forward pass to see if it produces the expected output shape and values. You can modify the input feature matrix X and the adjacency matrix A to test with different graphs and features or add more layers to create a deeper GCN model. Additionally, you can implement a training loop to learn the weights of the GCN layer using a loss function and an optimizer, but for now we are just focusing on the forward pass to understand how the GCN layer works.

# Example Usage
input_dim = 3  # Assuming the input dimension is 3
output_dim = 2  # Assuming the output dimension is 2

# Example adjacency matrix
A = torch.tensor([[1., 0., 0.],
                    [0., 1., 1.],
                    [0., 1., 1.]])  

# Create the GCN Layer
gcn_layer = GCNLayer(input_dim, output_dim, A)

# Example input feature matrix
X = torch.tensor([[1., 2., 3.],
                    [4., 5., 6.],
                    [7., 8., 9.]])

# Forward pass
output = gcn_layer(X)

print(output)

tensor([[ 6.3714,  9.5829],
        [14.9315, 22.1649],
        [17.4698, 25.9093]], grad_fn=<ReluBackward0>)


## More Complex GCN Code from PyTorch

An example code for using the GCN function from PyTorch/Torch Geometric. The base setup is similar to Monday's code but this time we are doing link prediction instead of node classification.

In [10]:
#############
## IMPORTS ##
#############
import os
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T
from torch_geometric.nn import GCNConv

###################
## DEVICE SET UP ##
###################
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
#######################
## CREATE MODEL: GCN ##
#######################
# The GCN model consists of two GCNConv layers with a ReLU activation and dropout in between. The first layer takes the input 
# features and transforms them to a hidden representation, while the second layer takes the hidden representation and produces 
# the final output features The dropout is used to prevent overfitting during training. You can adjust the number of layers,
#  hidden dimensions, and dropout rate as needed for your specific task and dataset.
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.3):
        """
        Inputs:
            in_channels: The number of features in the input node feature matrix X. This is the dimensionality of the input 
            node features.
            hidden_channels: The number of features in the hidden layer. This is the dimensionality of the hidden node features 
            after the first GCN layer.
            out_channels: The number of features in the output node feature matrix. This is the dimensionality of the output node 
            features after the second GCN layer, which is typically the number of classes for node classification tasks.
            dropout: The dropout rate to apply after the first GCN layer. This is a value between 0 and 1 that determines the 
            probability of dropping out nodes during training to prevent overfitting. A common value is 0.3, which means that 30% of 
            the nodes will be randomly dropped out during training.
        Outputs:
            None
        Initializes the GCN model by setting up the necessary layers and parameters for the forward pass.
        """
        # Initialize the parent class.
        super().__init__()
        # Define the first GCN layer that takes in the input features and transforms them to the hidden representation. The in_channels parameter 
        # specifies the number of input features, and the hidden_channels parameter specifies the number of output features for this layer.
        self.conv1 = GCNConv(in_channels, hidden_channels)
        # Define the second GCN layer that takes in the hidden representation and transforms it to the final output features. The hidden_channels 
        # parameter specifies the number of input features for this layer (which is the output of the first layer), and the out_channels parameter 
        # specifies the number of output features for this layer.
        self.conv2 = GCNConv(hidden_channels, out_channels)
        # Set the dropout rate for the model. This will be used in the forward pass to apply dropout after the first GCN layer.
        self.dropout = dropout

    def forward(self, x, edge_index):
        """
        Inputs:
            x: The input node feature matrix. This is a matrix where each row corresponds to a node in the graph, and each column corresponds to a feature of that node. The shape of x is (num_nodes, in_channels).
            edge_index: The edge index tensor that defines the graph structure. This is a tensor that specifies the connections between nodes in the graph. It typically has a shape of (2, num_edges), where the first row contains the source nodes and the second row contains the target nodes for each edge.
        Outputs:
            The output node feature matrix after applying the GCN layers. This is a matrix where each row corresponds to a node in the graph, and each column corresponds to a feature of that node after the GCN layers. The shape of the output is (num_nodes, out_channels).
        Forward pass of the GCN model. This is where the actual computation happens. The input x is the node feature matrix, and the output is the new node feature matrix after applying the two GCN layers with ReLU activation and dropout in between.
        """
        # Send the input node features x and the edge index to the first GCN layer. This will produce a new node feature matrix where each node's features 
        # have been transformed based on its neighbors' features and the graph structure defined by edge_index.
        x = self.conv1(x, edge_index)
        # Apply ReLU activation to the output of the first GCN layer. This introduces non-linearity to the model, allowing it to learn more complex 
        # patterns in the data.
        x = F.relu(x)
        # Apply dropout to the output of the ReLU activation. This randomly drops out nodes during training with a probability defined by self.dropout to 
        # prevent overfitting. The training parameter ensures that dropout is only applied during training and not during evaluation.
        x = F.dropout(x, p=self.dropout, training=self.training)
        # Send the output of the dropout layer and the edge index to the second GCN layer. This will produce the final output node feature matrix. For link
        # prediction task we will not have an activation function here (like for node classificaiton) but will instead use a later decoding funciton.
        x = self.conv2(x, edge_index)
        return x

In [12]:
##########################################
## DECODER FUNCTION FOR LINK PREDICTION ##
###########################################
def decode(z, edge_label_index):
    """
    Inputs:
        z: The output node feature matrix from the GCN model. The shape of z is (num_nodes, out_channels).
        edge_label_index: The edge index tensor that defines the pairs of nodes for which we want to predict the links.
    Outputs:
        A tensor of logits for each pair of nodes defined in edge_label_index. The shape of the output is (num_edges,), where each entry corresponds to the logit for the presence of a link between the source and target nodes specified in edge_label_index.
    The decode function takes the output node feature matrix z from the GCN model and the edge_label_index, which specifies the pairs of nodes for which we want to predict the links. The function computes the logits for each pair of nodes by taking the dot product of their corresponding feature vectors in z. The edge_label_index is typically a tensor of shape (2, num_edges) where the first row contains the source nodes and the second row contains the target nodes for each edge. The function returns a tensor of logits that can be used for link prediction, where higher logits indicate a higher likelihood of a link existing between the corresponding nodes.
    """
    # edge_label_index: [2, num_edges] of (src, dst)
    src, dst = edge_label_index
    return (z[src] * z[dst]).sum(dim=-1)  # dot product logits

##############################
## GET LINK LOGITS FUNCTION ##
##############################
def get_link_logits(model, data):
    """
    Inputs:
        model: The GCN model that has been trained or is being evaluated. This model should be an instance of the GCN class defined earlier, and it should have its parameters set (either through training or loading a pre-trained model).
        data: The graph data object that contains the node features, edge index, and edge label index. This is typically an instance of a PyTorch Geometric data class (e.g., Data) that includes the necessary information for the GCN model to perform link prediction.
    Outputs:
        A tensor of logits for each pair of nodes defined in data.edge_label_index. The shape of the output is (num_edges,), where each entry corresponds to the logit for the presence of a link between the source and target nodes specified in data.edge_label_index.
    The get_link_logits function takes a GCN model and a graph data object as input. It first passes the node features and edge index from the data through the GCN model to obtain the output node feature matrix z. Then, it uses the decode function to compute the logits for each pair of nodes specified in data.edge_label_index. The resulting logits can be used for link prediction tasks, where higher logits indicate a higher likelihood of a link existing between the corresponding nodes.
    """
    z = model(data.x, data.edge_index)
    logits = decode(z, data.edge_label_index)
    return logits


In [13]:

#######################
## TRAINING FUNCTION ##
#######################
def train(model, optimizer, train_data):
    """
    Inputs:
        model: The GCN model that we want to train. This should be an instance of the GCN class defined earlier, and it should have its parameters initialized (either randomly or from a pre-trained state).
        optimizer: The optimizer that will be used to update the model's parameters during training. 
        train_data: The graph data object that contains the node features, edge index, edge label index, and edge labels for the training set. This should be an instance of a PyTorch Geometric data class (e.g., Data) that includes the necessary information for training the GCN model on the link prediction task or a handmade dataset that can be used in the same way.
    Outputs:
        The loss value for the current training iteration. This is a scalar value that represents the binary cross-entropy loss computed between the predicted logits for the link prediction task and the true edge labels in train_data.edge_label. 
    The train function is responsible for performing one iteration of training for the GCN model on the link prediction task. 
    """
    model.train()
    optimizer.zero_grad()
    # This acts as the forward pass since the function passes the data through the model and then decodes the output to get the logits for the link prediction task. The logits are then used to compute the binary cross-entropy loss against the true edge labels, which indicates whether a link exists or not between the pairs of nodes specified in train_data.edge_label_index. The loss is then backpropagated to update the model's parameters using the optimizer.
    logits = get_link_logits(model, train_data)
    loss = F.binary_cross_entropy_with_logits(logits, train_data.edge_label.float())

    loss.backward()
    optimizer.step()
    return loss.item()

##############################################################
## PREDICTION FUNCTION FOR LINK PREDICTION (ACCURACY BASED) ##
##############################################################
@torch.no_grad()
def accuracy(model, data, threshold=0.5):
    """"
    Inputs:
        model: The GCN model that we want to evaluate. This should be an instance of the GCN class defined earlier, and it should have its parameters set (either through training or loading a pre-trained model).
        data: The graph data object that contains the node features, edge index, edge label index, and edge labels for the evaluation set. This is typically an instance of a PyTorch Geometric data class (e.g., Data) that includes the necessary information for evaluating the GCN model on the link prediction task.
        threshold: The threshold value for converting predicted probabilities into binary predictions. This is a float value between 0 and 1 that determines the cutoff point for classifying a predicted logit as indicating the presence of a link (1) or the absence of a link (0). A common default value is 0.5, which means that if the predicted probability is greater than or equal to 0.5, it will be classified as a link (1), and if it is less than 0.5, it will be classified as no link (0).
    Outputs:
        The accuracy of the model's predictions for the link prediction task. This is a float value between 0 and 1 that represents the proportion of correct predictions made by the model. The accuracy is calculated by comparing the binary predictions (after applying the threshold to the predicted probabilities) with the true edge labels in data.edge_label. The accuracy is computed as the number of correct predictions divided by the total number of predictions, giving a measure of how well the model is performing on the link prediction task
    The accuracy function is responsible for evaluating the performance of the GCN model on the link prediction task. 
    """
    model.eval()
    logits = get_link_logits(model, data)

    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).long()
    labels = data.edge_label.long()

    acc = (preds == labels).float().mean().item()
    return acc




In [15]:
#####################
## IMPORT THE DATA ##
#####################

# Transforms for the Cora data set. We use NormalizeFeatures to normalize the node features and RandomLinkSplit to split the edges into training, validation, and test sets for the link prediction task. The RandomLinkSplit transform also adds negative samples for training, which are pairs of nodes that do not have an edge between them. The num_val and num_test parameters specify the proportion of edges to be used for validation and testing, respectively. The is_undirected parameter indicates that the graph is undirected, meaning that if there is an edge from node A to node B, there is also an edge from node B to node A. The add_negative_train_samples parameter specifies that negative samples should be added to the training set, and the neg_sampling_ratio parameter determines how many negative samples to add relative to the number of positive samples (edges) in the training set.
# Split edges into train/val/test + negative samples for training
transform = T.Compose([
    T.NormalizeFeatures(),
    T.RandomLinkSplit(
        num_val=0.05,
        num_test=0.10,
        is_undirected=True,
        add_negative_train_samples=True,
        neg_sampling_ratio=1.0
    )
])
# Import the data set and split into trianing, validation, and test. Send the data to the device (GPU if available, otherwise CPU) for efficient computation during training and evaluation.
dataset = Planetoid(root="data/", name="Cora", transform=transform)
train_data, val_data, test_data = dataset[0]  # RandomLinkSplit returns 3 Data objects

train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)

#####################
## MODEL CREATTION ##
#####################

# Create a GCN instance with the appropriate input and output dimensions based on the dataset. The in_channels is set to the number of node features in the dataset, hidden_channels is set to 64 (you can adjust this as needed), and out_channels is also set to 64 . The dropout rate is set to 0.3, which means that 30% of the nodes will be randomly dropped out during training to prevent overfitting.
model = GCN(
    in_channels=dataset.num_node_features,
    hidden_channels=64,
    out_channels=64,
    dropout=0.3
).to(device)

# Set up the optimizer for training the GCN model. We are using the Adam optimizer, which is a popular choice for training neural networks. The learning rate is set to 0.01, and weight decay is set to 5e-4, which helps to prevent overfitting by adding a regularization term to the loss function that penalizes large weights. You can adjust the learning rate and weight decay as needed for your specific task and dataset.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

##################################
## TRAINING AND EVALUATION LOOP ##
##################################

# Values to keep track of the best validation accuracy at that point during training. 
best_val = 0.0

# Training loop for 200 epochs. 
for epoch in range(1, 201):
    # Train the model and then get the validation accuracy
    loss = train(model, optimizer, train_data)
    val_acc = accuracy(model, val_data, threshold=0.5)

    # Update best validation accuracy if current validation accuracy is higher than the best validation accuracy seen so far during training. This allows us to keep track of the best performance of the model on the validation set, which is important for model selection and early stopping.
    if val_acc > best_val:
        best_val = val_acc

    # Every 20 epochs print information to the console to track the training.
    if epoch % 20 == 0:
        print("Epoch:", epoch,  "Loss:" ,loss, "Val Acc:", val_acc*100,"%")

print("\nBest validation accuracy:", round(best_val, 4)*100, "%")

test_acc = accuracy(model, test_data, threshold=0.5)
print("Test accuracy at best val:", round(test_acc, 4)*100, "%")




Epoch: 20 Loss: 0.65366131067276 Val Acc: 50.57034492492676 %
Epoch: 40 Loss: 0.5721152424812317 Val Acc: 65.20912647247314 %
Epoch: 60 Loss: 0.5433288812637329 Val Acc: 62.92775869369507 %
Epoch: 80 Loss: 0.5380668640136719 Val Acc: 63.30798268318176 %
Epoch: 100 Loss: 0.537723958492279 Val Acc: 63.117873668670654 %
Epoch: 120 Loss: 0.5374523997306824 Val Acc: 61.596959829330444 %
Epoch: 140 Loss: 0.5374741554260254 Val Acc: 63.117873668670654 %
Epoch: 160 Loss: 0.5368948578834534 Val Acc: 63.117873668670654 %
Epoch: 180 Loss: 0.5339176654815674 Val Acc: 61.596959829330444 %
Epoch: 200 Loss: 0.53505939245224 Val Acc: 62.167298793792725 %

Best validation accuracy: 65.4 %
Test accuracy at best val: 67.36 %


## Resources
* [Demystifying GCNs: A Step-by-Step Guide to Building a Graph Convolutional Network Layer in PyTorch](https://medium.com/@jrosseruk/demystifying-gcns-a-step-by-step-guide-to-building-a-graph-convolutional-network-layer-in-pytorch-09bf2e788a51)
* [A Gentle Introduction to Graph Neural Networks](https://distill.pub/2021/gnn-intro/)
* [Understanding Convolutions on Graphs](https://distill.pub/2021/understanding-gnns/)
* [Graph Convolutional Networks: Introduction to GNNs](https://medium.com/data-science/graph-convolutional-networks-introduction-to-gnns-24b3f60d6c95)
* [Graph Convolutional Networks](https://tkipf.github.io/graph-convolutional-networks/)
* [Original Paper](https://arxiv.org/pdf/1609.02907)
* [Graph Convolutional Networks - Oxford Geometric Deep Learning (Video)](https://www.youtube.com/watch?v=CwHNUX2GWvE)
* [Graph Neural Networks - a perspective from the ground up (Video)](https://www.youtube.com/watch?v=GXhBEj1ZtE8)
* [Graph Convolutional Networks (GCNs) made simple (Video)](https://www.youtube.com/watch?v=2KRAOZIULzw)
* [Graph Convolutional Networks (GCN) | GNN Paper Explained (Video)](https://www.youtube.com/watch?v=VyIOfIglrUM)



